In [ ]:
# Part 1
# Paste the output of the --list command here as a comment:
# __consumer_offsets
# lab1
# transactions

## Part 2: Producer – Generating Transactions
### Task 2.1 – Write a transaction producer

In [ ]:
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='localhost:29092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

tx_counter = 1

def generate_transaction():
    global tx_counter

    transaction = {
        "tx_id": f"TX{tx_counter:04d}",
        "user_id": f"u{random.randint(1, 20):02d}",
        "amount": round(random.uniform(5.0, 5000.0), 2),
        "store": random.choice(["Warsaw", "Krakow", "Gdansk", "Wroclaw"]),
        "category": random.choice(["electronics", "clothing", "food", "books"]),
        "timestamp": datetime.now().isoformat()
    }

    tx_counter += 1
    return transaction

for _ in range(50):
    tx = generate_transaction()
    producer.send("transactions", tx)
    print(tx)
    time.sleep(1)

producer.flush()

## Part 3: Stateless Consumer – Filtering
### Task 3.1 – Filter large transactions

In [29]:
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='localhost:29092',
    auto_offset_reset='earliest',
    group_id='filter-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Listening for large transactions (amount > 1000)...")

for message in consumer:
    tx = message.value

    if tx["amount"] > 1000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']} PLN | {tx['store']} | {tx['category']}")

Listening for large transactions (amount > 1000)...


KeyboardInterrupt: 

### Task 3.2 – Transform and enrich

In [ ]:
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='localhost:29092',  # IMPORTANT
    auto_offset_reset='earliest',
    group_id='enrich-group',  # MUST be different!
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Listening and enriching transactions...")

for message in consumer:
    tx = message.value
    
    # Add risk_level
    if tx["amount"] > 3000:
        tx["risk_level"] = "HIGH"
    elif tx["amount"] > 1000:
        tx["risk_level"] = "MEDIUM"
    else:
        tx["risk_level"] = "LOW"
    
    print(tx)

## Part 4: Stateful Consumer – Aggregating